# Chapter 3: RAG and Retrieval Evaluation

Estimated time: about 8 hours (the longest chapter in this course).

Prerequisites: Chapters 1-2 (`agentlib.llm_client`, the real/mock brain toggle).

Interview category this chapter maps to: RAG evaluation and diagnosis, such as "is this a
retrieval problem or a generation problem, and which metric proves it," chunking strategy for
messy real-world corpora, and explaining hallucination-despite-grounding to a non-technical
stakeholder.

## Setup

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import random
import re

import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from agentlib import eval_metrics, llm_client, synthetic_data
from agentlib.grading import check

random.seed(42)
np.random.seed(42)

nlp = spacy.load("en_core_web_md")

print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")


## Section 1: Definitions

### Three failure surfaces in a RAG pipeline

A RAG pipeline has three stages that fail independently. Diagnosing which stage broke is one
of the most common interview questions in this space.

**Retrieval failure:** the right document exists in the corpus but was not returned in the
top-k results. A Zendesk support bot that returns billing FAQ articles when a customer asks
about password resets has a retrieval failure. The password-reset article exists in the index
but the retriever ranked it below billing content. Precision@k drops to zero for that query.

**Ranking failure:** the right document was retrieved but buried below less-relevant ones.
A medical literature search retrieves a drug-interaction warning but places it at position
15 out of 20 results. The clinician reads the top 3 and misses the warning entirely. MRR
drops; recall@k may still look fine if the cutoff is generous enough to include position 15.

**Generation failure:** the right document was retrieved and ranked well, but the model
ignored it and answered from its own parametric knowledge. A legal research AI retrieves
the correct case precedent at rank 1 and then generates an answer that contradicts it.
Every retrieval metric is perfect; a faithfulness check is what catches this failure.

Each stage requires a different metric. Precision@k and recall@k diagnose retrieval. MRR
diagnoses ranking. A faithfulness or groundedness score diagnoses generation. This chapter
builds all three retrieval metrics and demonstrates the generation failure hands-on.

### Retrieval metrics

**Precision@k**: of the top-k documents retrieved, what fraction were relevant?
You retrieve 5 documents and 2 are relevant: precision@5 = 2/5 = 0.4. A retriever that pads
the list with irrelevant results is penalized here.

**Recall@k**: of all relevant documents in the corpus, what fraction appeared in the top-k?
3 relevant documents exist. Your top-5 contains 2 of them: recall@5 = 2/3. A retriever that
finds one perfect document and misses four others is penalized here.

**MRR (Mean Reciprocal Rank)**: 1 / rank of the first relevant result. Rank 1 scores 1.0,
rank 2 scores 0.5, rank 5 scores 0.2, not found scores 0. Averaged across all queries in a
test set. MRR cares only about where the first hit lands, not how many relevant documents
exist in total.

Google's search quality team uses variants of these metrics to evaluate ranking changes
before shipping them to production. Netflix uses recall-based metrics to measure
recommendation coverage across its catalog.

### Chunking strategies

Fixed-size chunking splits at a raw character count with no regard for document structure.
Fast and predictable, but it will cut a sentence or a code block in half.

Semantic chunking splits at natural boundaries: section headers, paragraph breaks, sentence
endings. Each chunk is a coherent unit, but chunks vary in size and the splitting logic is
domain-specific.

Overlap lets consecutive chunks share a region of text. Any fact shorter than the overlap
sits fully inside at least one chunk. The cost is index size: halving the step roughly
doubles the chunk count.

Rajpurkar et al. (2016) found that over 15% of SQuAD answer spans cross sentence boundaries,
which is exactly the class of fact that overlap recovers.

### BM25 vs. TF-IDF vs. embeddings

Three retrieval approaches with different failure modes.

**TF-IDF** weights a term by its frequency in this document relative to its rarity across
all documents. Fast and interpretable. Fails on synonyms: "car" and "automobile" score zero
similarity because TF-IDF operates on exact token matches.

**BM25** refines TF-IDF with two fixes. Term frequency saturates (a word appearing 40 times
does not contribute 40x), and document length is normalized (a sprawling document cannot win
on sheer volume). Elasticsearch uses BM25 as its default ranking function.

**Embedding retrieval** maps text to dense vectors where geometric distance corresponds to
meaning similarity. Handles synonyms and paraphrases well. Fails on arbitrary identifiers
(SKU-1234, part numbers, internal codes) that appeared in no training corpus and have no
learned vector representation.

The Elasticsearch documentation recommends BM25 for exact-match queries and dense retrieval
for semantic queries, with reciprocal rank fusion for hybrid pipelines. The precondition
(demonstrated in this chapter) is that both retrievers must be individually strong and fail
on different queries.

## Section 2: Concept Explanation

### RAG pipeline architecture

```
  Query
    |
    v
+------------+     Did we find the       precision@k
| Retrieval  |     right document(s)?    recall@k
+------------+
    |
    v
+------------+     Is the best one       MRR
|  Ranking   |     near the top?
+------------+
    |
    v
+------------+     Did the model         faithfulness
| Generation |     actually use it?      score
+------------+
    |
    v
  Answer
```

Each stage has its own failure mode and its own metric. A fix at the wrong stage wastes
effort: re-tuning the generator when retrieval is broken, or expanding the corpus when
ranking is the bottleneck.

### Failure mode diagnosis table

| Symptom | Retrieval metrics | Faithfulness | Broken stage | Fix |
|---|---|---|---|---|
| Confident wrong answer | precision@k = 0 | n/a | Retrieval | Better indexing, query expansion |
| Right doc buried | recall@k OK, MRR low | n/a | Ranking | Reranker, adjust scoring |
| Ignores retrieved context | All high | Low | Generation | Prompt engineering, faithfulness filter |
| Half-right answer | recall@k partial | Moderate | Retrieval or chunking | Raise k, add overlap, reranker |
| Subtly wrong fact | All look fine | High (spurious) | Confusable in corpus | Deduplication at ingest time |

### Why RAG does not eliminate hallucination

Retrieval places relevant material in front of the model. Nothing forces the model to use
it faithfully. The model can ignore retrieved context and answer from parametric knowledge,
or it can selectively use parts of the context while hallucinating others. This chapter
demonstrates that gap directly in Break It #2.

### Trade-offs

**top-k size:** larger k means better recall (more relevant documents included) but higher
cost (more tokens in the generation context) and more noise (irrelevant documents dilute
the signal).

**chunk size:** larger chunks preserve more context within each chunk but produce fewer,
coarser chunks. Smaller chunks give finer retrieval granularity but risk splitting facts
across boundaries.

**overlap:** more overlap means better recovery of boundary-spanning facts but larger index
size. With zero overlap, the index is smallest but any fact on a boundary is lost.

**hybrid search (BM25 + dense):** handles both exact-match and semantic queries, but fusing
a strong retriever with a weak one can move results toward the weak one. The only way to
know is to measure on your own query distribution.

## Section 3: Example Code Segments

Data loading, chunking, retrieval, and generation helpers used by the graded tasks and
experiments below.

### Real corpus from SQuAD 1.1

The base corpus is 25-30 real passages from SQuAD 1.1 (Rajpurkar et al., 2016), which ships
human-annotated ground-truth question/answer pairs mapped to each passage.
`agentlib.synthetic_data.load_squad_sample()` loads from a committed local cache
(`data/rag_corpus/squad_sample.json`) so no notebook run needs network access.

In [ ]:
squad = synthetic_data.load_squad_sample()
docs = squad["docs"]
qa_pairs = squad["qa_pairs"]

print(f"Loaded {len(docs)} real SQuAD passages and {len(qa_pairs)} real QA pairs.")
print(f"Source: {squad['source']}")
print(f"License: {squad['license']}")
print()
print("Example passage:", docs[0]["title"])
print(" ", docs[0]["text"][:200], "...")
print("Example QA:", qa_pairs[0]["question"], "->", qa_pairs[0]["answer"])


Loaded 28 real SQuAD passages and 55 real QA pairs.
Source: SQuAD 1.1 dev set (Rajpurkar et al., 2016), official rajpurkar/SQuAD-explorer GitHub repo
License: CC BY-SA 4.0 (inherited from underlying Wikipedia content; see REFERENCES.md)

Example passage: Genghis_Khan
  The Shah's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. This fragmentation was decisive in Khwarezmia's d ...
Example QA: What feature of the Shah's army enable the weary Mongol forces easy early victories? -> fragmentation


### Confusable documents

4-6 near-duplicate documents layered synthetically on top of the real corpus. Each is a real
passage with exactly one fact swapped, similar enough to be retrieved in place of the
original, wrong if you check the swapped fact. Built deterministically by
`agentlib.synthetic_data.generate_confusable_documents()`.

In [ ]:
confusables = synthetic_data.generate_confusable_documents(docs, n=5, seed=42)
all_docs = docs + confusables

print(f"Generated {len(confusables)} confusable documents. Corpus is now {len(all_docs)} docs.")
for c in confusables[:2]:
    original = next(d for d in docs if d["doc_id"] == c["confusable_of"])
    print(f"\n{c['confusable_of']} (original):   {original['text'][:120]}...")
    print(f"{c['doc_id']} (confusable): {c['text'][:120]}...")
    print(f"  synthetic change: {c['synthetic_change']}")


Generated 5 confusable documents. Corpus is now 33 docs.

squad-027 (original):   In anglophone academic works, theories regarding imperialism are often based on the British experience. The term "Imperi...
squad-027-confusable (confusable): In anglophone academic works, theories regarding imperialism are often based on the British experience. The term "Imperi...
  synthetic change: name 'Minister Benjamin' -> 'Sarah Thompson'

squad-003 (original):   Genghis Khan is regarded as one of the prominent leaders in Mongolia's history. He is responsible for the emergence of t...
squad-003-confusable (confusable): Sarah Thompson is regarded as one of the prominent leaders in Mongolia's history. He is responsible for the emergence of...
  synthetic change: name 'Genghis Khan' -> 'Sarah Thompson'


### Ingesting a genuinely messy real document

Retrieval quality is only as good as what gets indexed. Real-world documents are rarely
clean paragraphs. This uses `anthropic-sdk-python`'s actual `CHANGELOG.md` as a chunking
exercise: it has inconsistent section lengths, embedded links and commit hashes, and many
genuinely near-duplicate release-note sections.

In [ ]:
messy_text = synthetic_data.load_messy_corpus()
print(f"Loaded {len(messy_text)} characters / {len(messy_text.splitlines())} lines of real, unedited changelog text.")
print()
print(messy_text[:600])
print("...")


Loaded 50745 characters / 900 lines of real, unedited changelog text.

# Changelog

## 0.121.0 (2026-08-07)

Full Changelog: [v0.120.2...v0.121.0](https://github.com/anthropics/anthropic-sdk-python/compare/v0.120.2...v0.121.0)

### Features

* **api:** add `mid-conversation-tool-changes-2026-07-01` beta ([c7d1531](https://github.com/anthropics/anthropic-sdk-python/commit/c7d1531d9d63a35430333039f8c975bba4ef0411))
* **api:** add support for session budgets, advisor tool, pinned inference location and skills auto-loading from GitHub ([193bae0](https://github.com/anthropics/anthropic-sdk-python/commit/193bae02806219047d484d7efcb9b91776a6c45c))


### Chores

* **api:
...


### Chunking strategy: fixed-size vs. semantic

Fixed-size chunking splits on a raw character count, with no regard for document structure.
Semantic (section-boundary) chunking splits on natural boundaries (here, `##`/`###` Markdown
headers). Each chunk is a coherent unit. Both are worked side by side against the same messy
input.

In [ ]:
def chunk_fixed_size(text: str, chunk_size: int = 400) -> list:
    '''Splits on a raw character count -- fast, simple, structure-blind.'''
    return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]


def chunk_by_section(text: str) -> list:
    '''Splits on Markdown section headers (## or ###) -- respects document structure, so a
    chunk is never a sentence or code block cut in half.'''
    sections = re.split(r"\n(?=##+\s)", text)
    return [s.strip() for s in sections if s.strip()]


fixed_chunks = chunk_fixed_size(messy_text)
section_chunks = chunk_by_section(messy_text)

print(f"Fixed-size chunking:  {len(fixed_chunks)} chunks")
print(f"Semantic (section) chunking: {len(section_chunks)} chunks")
print()
print("--- Fixed-size chunk #3 (notice it starts/ends mid-sentence) ---")
print(repr(fixed_chunks[3][:250]))
print()
print("--- Semantic chunk #3 (starts cleanly at a section header) ---")
print(repr(section_chunks[3][:250]))


Fixed-size chunking:  127 chunks
Semantic (section) chunking: 188 chunks

--- Fixed-size chunk #3 (notice it starts/ends mid-sentence) ---
'[d52999b](https://github.com/anthropics/anthropic-sdk-python/commit/d52999b4f66002ffe8263756c59ab0e276c74264))\n\n## 0.120.2 (2026-07-28)\n\nFull Changelog: [v0.120.1...v0.120.2](https://github.com/anthropics/anthropic-sdk-python/compare/v0.120.1...v0.12'

--- Semantic chunk #3 (starts cleanly at a section header) ---
'### Chores\n\n* **api:** remove retired Claude Opus 4.1 models ([5352a33](https://github.com/anthropics/anthropic-sdk-python/commit/5352a33ade5b62ad782b522f1791b7eeeeff83be))\n* **docs:** small updates to descriptions ([b8d4176](https://github.com/anthr'


### Deduplicating near-identical chunks

This changelog has many release sections that are near-identical in structure. Indexing every
one wastes index space and can crowd out more distinctive chunks in retrieval. The filter
below catches near-duplicates using TF-IDF cosine similarity before they enter the index.

In [ ]:
def deduplicate_chunks(chunks: list, threshold: float = 0.85) -> list:
    '''Drops any chunk that's near-identical (TF-IDF cosine similarity above threshold) to a
    chunk already kept -- a real, working near-duplicate filter, not just a description of
    one.'''
    if not chunks:
        return []
    vectorizer = TfidfVectorizer().fit(chunks)
    vectors = vectorizer.transform(chunks)

    kept_indices = [0]
    for i in range(1, len(chunks)):
        sims = cosine_similarity(vectors[i], vectors[kept_indices])[0]
        if sims.max() < threshold:
            kept_indices.append(i)
    return [chunks[i] for i in kept_indices]


deduped_chunks = deduplicate_chunks(section_chunks)
print(f"Before dedup: {len(section_chunks)} section chunks")
print(f"After dedup:  {len(deduped_chunks)} chunks ({len(section_chunks) - len(deduped_chunks)} near-duplicates removed)")


Before dedup: 188 section chunks
After dedup:  184 chunks (4 near-duplicates removed)


### Two retrieval implementations

TF-IDF (via scikit-learn) and local embeddings (using spaCy's `en_core_web_md`). Both
implement the same `retrieve(query, k)` interface so they are directly comparable.

In [ ]:
class TfidfRetriever:
    def __init__(self, docs: list):
        self.doc_ids = [d["doc_id"] for d in docs]
        self.vectorizer = TfidfVectorizer()
        self.doc_vectors = self.vectorizer.fit_transform([d["text"] for d in docs])

    def retrieve(self, query: str, k: int = 3) -> list:
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.doc_vectors)[0]
        ranked = sims.argsort()[::-1][:k]
        return [self.doc_ids[i] for i in ranked]


class EmbeddingRetriever:
    def __init__(self, docs: list, nlp):
        self.doc_ids = [d["doc_id"] for d in docs]
        self.nlp = nlp
        self.doc_vectors = np.array([nlp(d["text"]).vector for d in docs])

    def retrieve(self, query: str, k: int = 3) -> list:
        query_vec = self.nlp(query).vector
        norms = np.linalg.norm(self.doc_vectors, axis=1) * np.linalg.norm(query_vec) + 1e-9
        sims = (self.doc_vectors @ query_vec) / norms
        ranked = sims.argsort()[::-1][:k]
        return [self.doc_ids[i] for i in ranked]


tfidf_retriever = TfidfRetriever(all_docs)
embedding_retriever = EmbeddingRetriever(all_docs, nlp)

sample_query = qa_pairs[0]["question"]
print("Query:", sample_query)
print("TF-IDF top 3:    ", tfidf_retriever.retrieve(sample_query, k=3))
print("Embedding top 3: ", embedding_retriever.retrieve(sample_query, k=3))
print("Gold doc:        ", qa_pairs[0]["gold_doc_id"])


Query: What feature of the Shah's army enable the weary Mongol forces easy early victories?
TF-IDF top 3:     ['squad-000', 'squad-000-confusable', 'squad-020']
Embedding top 3:  ['squad-000', 'squad-000-confusable', 'squad-015']
Gold doc:         squad-000


### FAISS vector index

Hand-rolled cosine similarity over a NumPy array works at this corpus's scale, but a real
system uses a proper vector index. Here is the same corpus indexed with FAISS (Johnson,
Douze & Jegou, 2017), which is what you would reach for in production.

In [ ]:
import faiss

embedding_matrix = embedding_retriever.doc_vectors.astype("float32")
faiss.normalize_L2(embedding_matrix)  # so inner product == cosine similarity

index = faiss.IndexFlatIP(embedding_matrix.shape[1])
index.add(embedding_matrix)

query_vec = nlp(sample_query).vector.astype("float32").reshape(1, -1)
faiss.normalize_L2(query_vec)
scores, indices = index.search(query_vec, k=3)

faiss_top_3 = [embedding_retriever.doc_ids[i] for i in indices[0]]
print("FAISS top 3:", faiss_top_3)
print("(matches the hand-rolled EmbeddingRetriever above, as it should -- same vectors, same metric)")


FAISS top 3: ['squad-000', 'squad-000-confusable', 'squad-015']
(matches the hand-rolled EmbeddingRetriever above, as it should -- same vectors, same metric)


### Catalog documents: identifier vs. paraphrase queries

`agentlib.retrieval_lab` has a small parts catalogue where every document carries an
identifier in its indexed text, and the queries are split into ones that need an exact match
and ones that are pure paraphrase. This tests the specific failure mode where embedding
retrieval cannot match an arbitrary serial number.

In [ ]:
from agentlib.retrieval_lab import (
    CATALOG_DOCS,
    IDENTIFIER_QUERIES,
    PARAPHRASE_QUERIES,
    tokenize,
)

for doc in CATALOG_DOCS[:3]:
    print(f"{doc['doc_id']}  {doc['text'][:78]}")
print(f"\n{len(CATALOG_DOCS)} documents")
print(f"{len(IDENTIFIER_QUERIES)} identifier queries, e.g. {IDENTIFIER_QUERIES[0][0]!r}")
print(f"{len(PARAPHRASE_QUERIES)} paraphrase queries, e.g. {PARAPHRASE_QUERIES[0][0]!r}")
print(f"\ntokenize('SKU-6690') -> {tokenize('SKU-6690')}")
print("Note the digits survive as their own token. A tokenizer that stripped them -- or")
print("stemmed aggressively -- would destroy the identifiers this corpus exists to test.")

SKU-4417  SKU-4417 ThermaFlow X2. Tankless electric unit that heats water on demand for 
SKU-9302  SKU-9302 ArcticCore 12K. Portable cooling appliance for a single room. Vents t
SKU-1188  SKU-1188 HelioPanel 400W. Monocrystalline photovoltaic module for rooftop moun

20 documents
6 identifier queries, e.g. 'SKU-6690'
8 paraphrase queries, e.g. 'stop my water lines bursting in a hard freeze'

tokenize('SKU-6690') -> ['sku', '6690']
Note the digits survive as their own token. A tokenizer that stripped them -- or
stemmed aggressively -- would destroy the identifiers this corpus exists to test.


### Generation: real model by default

Reusing `agentlib.llm_client` and the `HAS_KEY` toggle from Chapter 1. The generation step
calls a real model by default so you see actual model behavior when it ignores or misuses
retrieved context. Falls back to a template-based extractive generator if no API key is
present.

In [ ]:
def template_generate(query: str, retrieved_docs: list) -> str:
    '''Deterministic, template-based generator stand-in: picks the best-matching (highest
    query word-overlap) sentence from EACH retrieved doc and concatenates them -- crude, but
    free, explainable, fully offline, and it genuinely uses more of the context when more
    context is retrieved (unlike picking one single best sentence globally, which would
    ignore additional retrieved docs entirely).'''
    query_words = set(re.findall(r"[a-z0-9]+", query.lower()))

    def overlap(sentence):
        return len(query_words & set(re.findall(r"[a-z0-9]+", sentence.lower())))

    best_per_doc = []
    for doc in retrieved_docs:
        sentences = re.split(r"(?<=[.!?])\s+", doc["text"])
        best = max(sentences, key=overlap, default="").strip()
        if best:
            best_per_doc.append(best)
    return " ".join(best_per_doc)


def real_generate(query: str, retrieved_docs: list) -> str:
    context = "\n\n".join(d["text"] for d in retrieved_docs)
    response = llm_client.call_model(
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using ONLY the context above, in one sentence.",
        }],
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    return response.text


generate = real_generate if llm_client.HAS_KEY else template_generate
doc_lookup = {d["doc_id"]: d for d in all_docs}


def rag_answer(query: str, retriever, generate_fn, k: int = 3) -> dict:
    retrieved_ids = retriever.retrieve(query, k)
    retrieved_docs = [doc_lookup[doc_id] for doc_id in retrieved_ids]
    answer = generate_fn(query, retrieved_docs)
    return {"answer": answer, "retrieved_ids": retrieved_ids, "retrieved_docs": retrieved_docs}


demo_qa = qa_pairs[5]
result = rag_answer(demo_qa["question"], tfidf_retriever, generate)
print(f"Using: {'real model' if llm_client.HAS_KEY else 'template_generate (mock, no key present)'}")
print("Question:", demo_qa["question"])
print("Answer:  ", result["answer"])
print("Expected:", demo_qa["answer"])


Using: template_generate (mock, no key present)
Question: By which year did Chrysler ended its full sized luxury model?
Answer:   Chrysler ended production of their full-sized luxury sedans at the end of the 1981 model year, moving instead to a full front-wheel drive lineup for 1982 (except for the M-body Dodge Diplomat/Plymouth Gran Fury and Chrysler New Yorker Fifth Avenue sedans). However, it seems that if they had meant to call Genghis tenggis they could have said, and written, "Tenggis Khan", which they did not.) Zhèng (Chinese: 正) meaning "right", "just", or "true", would have received the Mongolian adjectival modifier -s, creating "Jenggis", which in medieval romanization would be written "Genghis". Most Western countries, and some others, have now banned it, but it remains lawful in the United States following a US Supreme Court decision in 1977 which held that paddling did not violate the US Constitution.
Expected: 1981


## Section 4: Build It Yourself

Five graded tasks: fixed-size chunking with overlap, the three core retrieval metrics
(precision@k, recall@k, MRR), and BM25 scoring. Each cell ends with
`your_function = check("task-id", your_function)` which runs a suite of assertions. Run
`python grade.py` at any point to see where you stand.

### Task 1: `chunk_with_overlap` (fixed-size chunking with overlap)

Both chunkers above cut the document at a boundary and move on, which means any fact that
straddles a boundary is split across two chunks. The standard fix is to let consecutive
chunks overlap: slide the window by less than its own width, so the last stretch of each
chunk is repeated at the head of the next.

**Hints:**

1. What happens to the last chunk? If you stop once the next full window would run past the
   end, the tail of every document becomes unretrievable.
2. Check `len(chunks[-1])` against `chunk_size`. The final partial chunk must be emitted.

In [ ]:
def chunk_with_overlap(text: str, chunk_size: int = 400, overlap: int = 0) -> list:
    '''Fixed-size chunks where each repeats the last `overlap` characters of the previous one.

    Step forward by `chunk_size - overlap` and slice `chunk_size` characters each time. With
    overlap=0 this must behave exactly like chunk_fixed_size above.

    Three things the cases check, all of them mistakes that look fine in a notebook and lose
    data in production:

    - No character may be dropped. Stepping by `chunk_size` while slicing `chunk_size -
      overlap` silently deletes the text in between.
    - The final partial chunk must be emitted. Stopping once the next full window would run
      past the end makes the tail of every document unretrievable.
    - overlap >= chunk_size means a step of zero or less, which never advances. Raise
      ValueError rather than hanging.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


chunk_with_overlap = check("ch03-chunk-overlap", chunk_with_overlap)

#### What overlap actually buys, measured

One worked example proves nothing: whether a particular fact survives a particular overlap is
mostly a question of where the boundaries happen to land. The honest measurement is
statistical -- place two related facts at random positions in many documents, and count how
often some single chunk contains both.

In [ ]:
def _recovery_rate(chunk_size, overlap, trials=800, seed=7):
    '''Fraction of trials where one chunk contains both facts, plus the mean chunk count.'''
    rng = random.Random(seed)
    hits = total_chunks = 0
    for _ in range(trials):
        length = rng.randint(600, 900)
        gap = rng.randint(40, 180)
        start = rng.randint(0, length - gap - 10)
        body = list("x" * length)
        body[start:start + 4] = "AAAA"
        body[start + gap:start + gap + 4] = "BBBB"
        chunks = chunk_with_overlap("".join(body), chunk_size, overlap)
        total_chunks += len(chunks)
        hits += any("AAAA" in c and "BBBB" in c for c in chunks)
    return hits / trials, total_chunks / trials


print(f"chunk_size=200, two related facts placed 40-180 characters apart\n")
print(f"{'overlap':>8}{'step':>6}{'chunks/doc':>12}{'both facts intact':>20}")
print("-" * 46)
for overlap in (0, 40, 80, 120, 160):
    rate, n_chunks = _recovery_rate(200, overlap)
    print(f"{overlap:>8}{200 - overlap:>6}{n_chunks:>12.1f}{rate:>19.1%}")

chunk_size=200, two related facts placed 40-180 characters apart

 overlap  step  chunks/doc   both facts intact
----------------------------------------------
       0   200         4.3              49.4%
      40   160         4.9              56.6%
      80   120         6.0              73.4%
     120    80         8.3              83.2%
     160    40        15.2              95.5%


That table is the tradeoff in one place: recovery climbs from under half to over 95%, and the
index grows about three and a half times to pay for it. Neither end is the right answer --
the right answer depends on how expensive your vectors are and how badly a split fact hurts.

Overlap and raising `k` fix the same problem differently. Overlap fixes it at index time, so
a single retrieved chunk is already complete. Raising `k` fixes it at query time, handing
the generator both halves and hoping it reassembles them. Overlap costs storage once; `k`
costs context window on every query. In practice you tune both.

### Retrieval evaluation harness

Three metrics, each answering a different question about the same ranked list:

- **precision@k** -- of what you showed the user, how much was worth showing?
- **recall@k** -- of what you should have found, how much did you actually find?
- **MRR** -- how far down did the user have to read before hitting something useful?

You write all three. `evaluate_retrieval` in `agentlib.eval_metrics` is the plumbing that
loops over queries and averages; it takes your functions as arguments, so the numbers below
are the ones your code produced.

### Task 2: `precision_at_k`

Watch the denominator. If you ask for `k=5` and the index only has 2 documents, dividing by
`k` penalizes the retriever for something it had no control over.

In [ ]:
def precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Of the top-k retrieved documents, what fraction are actually relevant?

    retrieved_ids -- doc_ids in rank order, best first
    relevant_ids  -- the doc_ids that genuinely answer this query
    k             -- how far down the ranking to look

    An empty top-k scores 0.0 rather than raising.
    """
    raise NotImplementedError("Implement me, then re-run this cell")


precision_at_k = check("ch03-precision-k", precision_at_k)

### Task 3: `recall_at_k`

Different denominator, and that difference is the whole distinction between the two metrics.
Recall asks about the documents that *should* have been found, including the ones your
retriever never returned at all.

In [ ]:
def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Of all the documents that were actually relevant, what fraction showed up in the
    top-k?

    A query with no relevant documents scores 0.0 rather than raising.
    """
    raise NotImplementedError("Implement me, then re-run this cell")


recall_at_k = check("ch03-recall-k", recall_at_k)

### Task 4: `mean_reciprocal_rank`

Reciprocal rank is a per-query score; averaging it across a query set is what makes it
*mean* reciprocal rank (which `evaluate_retrieval` does for you). Ranks are 1-based: the
top result is rank 1, not rank 0.

In [ ]:
def mean_reciprocal_rank(retrieved_ids: list, relevant_ids: set) -> float:
    """1 / (rank of the first relevant result), or 0.0 if none of the retrieved results are
    relevant. Ranks are 1-based.
    """
    raise NotImplementedError("Implement me, then re-run this cell")


mean_reciprocal_rank = check("ch03-mrr", mean_reciprocal_rank)

With all three metrics passing, run them over every real SQuAD question in the corpus and
compare the two retrievers head to head:

In [ ]:
eval_queries = [
    {"query": qa["question"], "relevant_doc_ids": {qa["gold_doc_id"]}}
    for qa in qa_pairs
]

metric_fns = {
    "precision_fn": precision_at_k,
    "recall_fn": recall_at_k,
    "mrr_fn": mean_reciprocal_rank,
}
tfidf_results = eval_metrics.evaluate_retrieval(eval_queries, tfidf_retriever.retrieve, k=3, **metric_fns)
embedding_results = eval_metrics.evaluate_retrieval(eval_queries, embedding_retriever.retrieve, k=3, **metric_fns)

print(f"Evaluated on {len(eval_queries)} real SQuAD questions, k=3\n")
print(f"{'metric':15s} {'TF-IDF':>10s} {'Embeddings':>12s}")
for metric in ["precision@k", "recall@k", "mrr"]:
    print(f"{metric:15s} {tfidf_results[metric]:>10.3f} {embedding_results[metric]:>12.3f}")


Evaluated on 55 real SQuAD questions, k=3

metric              TF-IDF   Embeddings
precision@k          0.315        0.206
recall@k             0.945        0.618
mrr                  0.861        0.536


### Task 5: `bm25_scores` (lexical retrieval)

Both retrievers above score documents by similarity in some vector space. Now that the eval
harness works, it can settle a question that matters in practice: what happens when the user
types an exact string?

TF-IDF was built for scoring documents; BM25 was built for ranking them, and the two
differences are worth knowing for interviews.

**Term frequency saturates.** In raw TF-IDF, a term appearing forty times contributes forty
times as much. BM25 routes tf through `tf * (k1 + 1) / (tf + k1 * ...)`, which rises quickly
at first and then flattens. A page that repeats a keyword cannot bury a document that uses it
twice in a genuinely relevant sentence.

**Length is normalised.** The `b` parameter scales the penalty by how far a document's length
deviates from the corpus average, so a sprawling document stops winning on sheer volume. At
`b=0` the penalty is off; at `b=1` it is full.

The `idf` term is the same idea as TF-IDF's: a word appearing in every document distinguishes
nothing, so its weight collapses toward zero.

In [ ]:
def bm25_scores(query: str, docs: list, k1: float = 1.5, b: float = 0.75) -> dict:
    '''Score every document against `query` with Okapi BM25. Return {doc_id: score}.

    For each query term present in a document:

        idf  = log(1 + (N - df + 0.5) / (df + 0.5))
        norm = 1 - b + b * (len(doc_tokens) / avg_doc_len)
        contribution = idf * tf * (k1 + 1) / (tf + k1 * norm)

    where N is the number of documents and df is how many of them contain the term at all.
    Sum the contributions. Use `tokenize` from agentlib.retrieval_lab so identifiers survive.

    Score EVERY document, including the ones that match nothing -- a caller has to be able to
    tell "scored zero" from "not in the index". Query terms absent from the whole corpus
    contribute nothing rather than crashing. Do not mutate `docs`.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


bm25_scores = check("ch03-bm25", bm25_scores)

### The comparison the eval harness now makes possible

Same catalogue, same queries, scored with the `mean_reciprocal_rank` you built earlier.

In [ ]:
def rank_all(score_fn, query):
    scores = score_fn(query, list(CATALOG_DOCS))
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda p: -p[1])]


catalog_ids = [d["doc_id"] for d in CATALOG_DOCS]
catalog_vectors = np.array([nlp(d["text"]).vector for d in CATALOG_DOCS])
_norms = np.linalg.norm(catalog_vectors, axis=1)


def dense_rank(query):
    qv = nlp(query).vector
    sims = (catalog_vectors @ qv) / (_norms * np.linalg.norm(qv) + 1e-9)
    return [catalog_ids[i] for i in sims.argsort()[::-1]]


def bm25_rank(query):
    return rank_all(bm25_scores, query)


print(f"{'query set':<14}{'n':>3}{'BM25 MRR':>11}{'dense MRR':>12}")
print("-" * 40)
for name, queries in (("identifier", IDENTIFIER_QUERIES), ("paraphrase", PARAPHRASE_QUERIES)):
    bm = np.mean([mean_reciprocal_rank(bm25_rank(q), {g}) for q, g in queries])
    dn = np.mean([mean_reciprocal_rank(dense_rank(q), {g}) for q, g in queries])
    print(f"{name:<14}{len(queries):>3}{bm:>11.3f}{dn:>12.3f}")

print("\nWhere the gold document ranks, per identifier query:")
for q, gold in IDENTIFIER_QUERIES:
    print(f"  {q:<14} BM25 #{bm25_rank(q).index(gold) + 1:<4} dense #{dense_rank(q).index(gold) + 1}")

query set       n   BM25 MRR   dense MRR
----------------------------------------
identifier      6      1.000       0.702


paraphrase      8      0.478       0.276

Where the gold document ranks, per identifier query:
  SKU-6690       BM25 #1    dense #15
  SKU-1273       BM25 #1    dense #7
  MERV 13        BM25 #1    dense #1
  GFCI           BM25 #1    dense #1
  ERV            BM25 #1    dense #1
  RO cartridge   BM25 #1    dense #1


### Read that carefully

BM25 gets a perfect 1.000 on the identifier queries. The dense retriever gets 0.702, and the
average hides the interesting part in the per-query breakdown.

The acronyms (`MERV 13`, `GFCI`, `ERV`, `RO cartridge`) are words: they occur in the text
the embedding model was trained on, so they have vectors that mean something. An arbitrary
serial number (`SKU-6690`, `SKU-1273`) does not and cannot. It was minted by a database, it
appeared in no training corpus, and the model has no representation for it beyond whatever the
tokenizer improvises. You cannot embed a fact the model has never seen; you can only match it
literally.

That is why the answer to "our vector search keeps missing exact product codes" is to add a
lexical index rather than to fine-tune the embedding model or lower the similarity threshold.

#### An honest caveat about the paraphrase row

BM25 also beats the dense retriever on the *paraphrase* queries here, which is not what you
would expect. The dense retriever in this chapter averages spaCy word vectors over a document.
That is a genuinely weak representation -- it has no way to compose meaning. A production
sentence-embedding model (sentence-transformers, Cohere embed, Voyage) would very likely win
this row.

This matters for a common claim: "combine BM25 with dense retrieval and fuse the scores."
That advice is sound, and it has a precondition people rarely state: **both retrievers have to
be individually strong, and they have to fail on different queries.** Fusing a strong retriever
with a weak one moves the result toward the weak one. The lesson is not that hybrid search does
not work; it is that "hybrid" is not a free upgrade, and the only way to know is to measure it
on your own corpus and query mix.

### Mapping this to RAGAS and DeepEval

The metrics above are hand-built so you understand exactly what they compute. In practice,
reach for a maintained library. RAGAS (Es et al., 2024) computes context precision/recall
analogous to precision@k/recall@k above, plus LLM-judged metrics like faithfulness. DeepEval
covers similar ground with a pytest-style testing interface, useful if you want
retrieval-quality regressions to fail CI like a broken unit test. Both are worth naming in an
interview.

## Section 5: Playground

Experiments with editable parameters. Change the values and re-run each cell to observe
the effect.

### Experiment 1: Chunk size

How does chunk size affect the number of chunks and their average length? Smaller chunks give
finer granularity but more index entries. Larger chunks preserve more context but are coarser.

In [ ]:
# --- EDIT THESE ---
CHUNK_SIZES = [200, 400, 1000]  # try [100, 300, 500, 2000]
# ------------------

print(f"Chunking {len(messy_text)} characters of changelog text:\n")
print(f"{'chunk_size':>11}{'chunks':>8}{'avg_len':>10}{'last_chunk':>12}")
print("-" * 41)
for cs in CHUNK_SIZES:
    chunks = chunk_fixed_size(messy_text, chunk_size=cs)
    avg_len = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{cs:>11}{len(chunks):>8}{avg_len:>10.0f}{len(chunks[-1]):>12}")

### Experiment 2: Overlap recovery rate

How does overlap affect the chance that two related facts land in the same chunk? Uses the
`_recovery_rate` helper from the Build section.

In [ ]:
# --- EDIT THESE ---
CHUNK_SIZE = 400
OVERLAPS = [0, 50, 100, 200]  # try [0, 80, 160, 240, 320]
# ------------------

print(f"chunk_size={CHUNK_SIZE}, varying overlap:\n")
print(f"{'overlap':>8}{'step':>6}{'chunks/doc':>12}{'recovery':>12}")
print("-" * 38)
for ov in OVERLAPS:
    rate, n = _recovery_rate(CHUNK_SIZE, ov)
    print(f"{ov:>8}{CHUNK_SIZE - ov:>6}{n:>12.1f}{rate:>11.1%}")

### Experiment 3: top-k and precision vs. recall

How does the choice of k trade off precision against recall? A large k improves recall (fewer
relevant documents are missed) but hurts precision (more irrelevant documents are included).

In [ ]:
# --- EDIT THESE ---
K_VALUES = [1, 3, 5, 10]  # try [1, 2, 3, 5, 10, 20]
# ------------------

print(f"Varying k on {len(eval_queries)} SQuAD queries, TF-IDF retriever:\n")
print(f"{'k':>4}{'precision@k':>14}{'recall@k':>12}{'MRR':>8}")
print("-" * 38)
for k in K_VALUES:
    results = eval_metrics.evaluate_retrieval(
        eval_queries, tfidf_retriever.retrieve, k=k, **metric_fns
    )
    print(f"{k:>4}{results['precision@k']:>14.3f}{results['recall@k']:>12.3f}{results['mrr']:>8.3f}")

### Experiment 4: BM25 vs. dense on your own queries

Try your own queries against both retrievers. Identifiers (SKU codes, acronyms) and
natural-language descriptions exercise different retrieval strengths.

In [ ]:
# --- EDIT THESE ---
TEST_QUERIES = [
    "SKU-6690",
    "stop my water lines bursting in a hard freeze",
    "MERV 13",
    "quiet cooling for a bedroom",
]
# ------------------

print(f"{'query':<48} {'BM25 #1':<18} {'Dense #1':<18}")
print("-" * 84)
for q in TEST_QUERIES:
    bm_top = bm25_rank(q)[0] if bm25_rank(q) else "n/a"
    dn_top = dense_rank(q)[0] if dense_rank(q) else "n/a"
    print(f"{q:<48} {str(bm_top):<18} {str(dn_top):<18}")

## Section 6: Break It

Five failure scenarios, each targeting a different stage of the RAG pipeline. Each is
demonstrated with the bug first and the fix second.

**Follow-up question (all scenarios):** "How do you know if your RAG system has a retrieval
problem or a generation problem?"

Answer: check retrieval metrics first. If precision@k and recall@k on the failing query are
healthy, retrieval is fine and the problem is downstream (ranking or generation). If
retrieval metrics are broken, no amount of prompt engineering fixes the answer because the
model never saw the right context.

### Break It 1: Corrupted retriever for a subset of documents

Simulating an index corruption or a bad deploy affecting some slice of traffic: for a
specific subset of documents, retrieval silently returns the wrong doc. Generation has no way
to know retrieval failed, so it produces a confident-sounding answer from the wrong context.

In [ ]:
CORRUPTED_DOC_IDS = {qa_pairs[i]["gold_doc_id"] for i in range(3)}
WRONG_REPLACEMENT_ID = all_docs[15]["doc_id"]


class CorruptedRetriever:
    def __init__(self, base_retriever):
        self.base = base_retriever

    def retrieve(self, query: str, k: int = 3) -> list:
        results = self.base.retrieve(query, k)
        return [WRONG_REPLACEMENT_ID if doc_id in CORRUPTED_DOC_IDS else doc_id for doc_id in results]


corrupted_retriever = CorruptedRetriever(tfidf_retriever)

print("--- Bug: corrupted retriever for a subset of docs ---\n")
buggy_eval = eval_metrics.evaluate_retrieval(eval_queries, corrupted_retriever.retrieve, k=3, **metric_fns)
healthy_eval = eval_metrics.evaluate_retrieval(eval_queries, tfidf_retriever.retrieve, k=3, **metric_fns)
print(f"Healthy precision@k/recall@k/mrr:   {healthy_eval['precision@k']:.3f} / {healthy_eval['recall@k']:.3f} / {healthy_eval['mrr']:.3f}")
print(f"Corrupted precision@k/recall@k/mrr: {buggy_eval['precision@k']:.3f} / {buggy_eval['recall@k']:.3f} / {buggy_eval['mrr']:.3f}")

affected_qa = qa_pairs[0]
buggy_result = rag_answer(affected_qa["question"], corrupted_retriever, generate)
print(f"\nExample -- Question: {affected_qa['question']}")
print(f"Retrieved (WRONG): {buggy_result['retrieved_ids']} -- should include {affected_qa['gold_doc_id']!r}")
print(f"Answer (confident but wrong): {buggy_result['answer']}")


--- Bug: corrupted retriever for a subset of docs ---



Healthy precision@k/recall@k/mrr:   0.315 / 0.945 / 0.861
Corrupted precision@k/recall@k/mrr: 0.291 / 0.873 / 0.797

Example -- Question: What feature of the Shah's army enable the weary Mongol forces easy early victories?
Retrieved (WRONG): ['squad-015', 'squad-000-confusable', 'squad-020'] -- should include 'squad-000'
Answer (confident but wrong): The Mongol military was also successful in siege warfare, cutting off resources for cities and towns by diverting certain rivers, taking enemy prisoners and driving them in front of the army, and adopting new ideas, techniques and tools from the people they conquered, particularly in employing Muslim and Chinese siege engines and engineers to aid the Mongol cavalry in capturing cities. Michael Chen's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. Later came The Clouded Yellow (1951) and Payroll (1961), both of which feature more extensive scenes fi

In [ ]:
print("--- Fix: restore correct retrieval ---\n")
fixed_eval = eval_metrics.evaluate_retrieval(eval_queries, tfidf_retriever.retrieve, k=3, **metric_fns)
print(f"Restored precision@k/recall@k/mrr: {fixed_eval['precision@k']:.3f} / {fixed_eval['recall@k']:.3f} / {fixed_eval['mrr']:.3f}")

fixed_result = rag_answer(affected_qa["question"], tfidf_retriever, generate)
print(f"\nRetrieved (correct): {fixed_result['retrieved_ids']}")
print(f"Answer: {fixed_result['answer']}")
print(f"Expected: {affected_qa['answer']}")


--- Fix: restore correct retrieval ---



Restored precision@k/recall@k/mrr: 0.315 / 0.945 / 0.861

Retrieved (correct): ['squad-000', 'squad-000-confusable', 'squad-020']
Answer: The Shah's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. Michael Chen's army was split by diverse internecine feuds and by the Shah's decision to divide his army into small groups concentrated in various cities. Later came The Clouded Yellow (1951) and Payroll (1961), both of which feature more extensive scenes filmed in the city.
Expected: fragmentation


### Break It 2: Retrieval succeeds, generation ignores it anyway

The failure the concept section warned about: nothing stops a generator from answering from
its own parametric knowledge even when the retrieved context is correct. Precision, recall,
and MRR are all fine here; the bug is invisible to retrieval metrics entirely. Faithfulness
is the metric that catches it.

In [ ]:
def parametric_generate(query: str, retrieved_docs: list) -> str:
    '''Deliberately ignores retrieved_docs and answers from a fixed, hardcoded response --
    the bug: retrieval succeeded, generation didn't use it.'''
    return "That's a well-established fact from general background knowledge."


print("--- Bug: retrieval is correct, generation ignores it ---\n")
qa = qa_pairs[10]
retrieved_ids = tfidf_retriever.retrieve(qa["question"], k=3)
print(f"Question: {qa['question']}")
print(f"Retrieved: {retrieved_ids} -- correct: {qa['gold_doc_id'] in retrieved_ids}")

retrieved_docs = [doc_lookup[d] for d in retrieved_ids]
bad_answer = parametric_generate(qa["question"], retrieved_docs)
bad_faithfulness = eval_metrics.faithfulness_score(bad_answer, [d["text"] for d in retrieved_docs])
print(f"Answer: {bad_answer!r}")
print(f"Faithfulness score: {bad_faithfulness:.2f}  <-- low, despite retrieval being correct")


--- Bug: retrieval is correct, generation ignores it ---

Question: What cultures were part of Kublai's administration?
Retrieved: ['squad-005', 'squad-008', 'squad-007'] -- correct: True
Answer: "That's a well-established fact from general background knowledge."
Faithfulness score: 0.43  <-- low, despite retrieval being correct


In [ ]:
print("--- Fix: generator actually uses the retrieved context ---\n")
good_answer = generate(qa["question"], retrieved_docs)
good_faithfulness = eval_metrics.faithfulness_score(good_answer, [d["text"] for d in retrieved_docs])
print(f"Answer: {good_answer!r}")
print(f"Faithfulness score: {good_faithfulness:.2f}")
print(f"\nSame retrieval both times ({retrieved_ids}) -- the only thing that changed is whether generation used it.")


--- Fix: generator actually uses the retrieved context ---

Answer: "Chinese advisers such as Liu Bingzhong and Yao Shu gave strong influence to Kublai's early court, and the central government administration was established within the first decade of Kublai's reign. Between 1402 and 1405, the expedition led by the Norman noble Jean de Bethencourt and the Poitevine Gadifer de la Salle conquered the Canarian islands of Lanzarote, Fuerteventura and El Hierro off the Atlantic coast of Africa. Retention rates for the final two years of secondary school were 77 per cent for public school students and 90 per cent for private school students."
Faithfulness score: 1.00

Same retrieval both times (['squad-005', 'squad-008', 'squad-007']) -- the only thing that changed is whether generation used it.


### Break It 3: Answer split across two chunks

Two short synthetic documents where neither alone answers the question, only their
combination does. Naive top-k retrieval with k=1 grabs only one half.

**Hint:** this is a different fix from chunk overlap. Overlap fixes boundary splits at index
time. Raising k fixes them at query time by retrieving both halves. They cost different
things: k costs context window on every query, overlap costs storage once.

In [ ]:
SPLIT_FACT_DOCS = [
    {"doc_id": "split-fact-A", "title": "Project Kestrel (part 1)",
     "text": "Project Kestrel was founded in 2019 as an internal research initiative."},
    {"doc_id": "split-fact-B", "title": "Project Kestrel (part 2)",
     "text": "Project Kestrel's founding team consisted of Priya Nair and Tom Alcock."},
]
split_fact_lookup = {d["doc_id"]: d for d in SPLIT_FACT_DOCS}
split_fact_retriever = TfidfRetriever(SPLIT_FACT_DOCS)

split_query = "Who founded Project Kestrel and in what year?"

print("--- Bug: k=1 only retrieves one half of the answer ---\n")
top_1 = split_fact_retriever.retrieve(split_query, k=1)
print(f"Retrieved (k=1): {top_1}")
partial_context = [split_fact_lookup[d] for d in top_1]
partial_answer = generate(split_query, partial_context)
print(f"Answer: {partial_answer!r}")
print("(only has the year, or only the founders -- never both, since only one chunk was retrieved)")


--- Bug: k=1 only retrieves one half of the answer ---

Retrieved (k=1): ['split-fact-A']
Answer: 'Project Kestrel was founded in 2019 as an internal research initiative.'
(only has the year, or only the founders -- never both, since only one chunk was retrieved)


In [ ]:
print("--- Fix: retrieve enough chunks (k=2) to cover both facts ---\n")
top_2 = split_fact_retriever.retrieve(split_query, k=2)
print(f"Retrieved (k=2): {top_2}")
full_context = [split_fact_lookup[d] for d in top_2]
full_answer = generate(split_query, full_context)
print(f"Answer: {full_answer!r}")
print("\nRaising k isn't a universal fix (it also raises noise and cost) -- the real lesson is")
print("that naive top-k retrieval has no way to know an answer needs multiple chunks at all;")
print("production systems handle this with larger k plus a reranker, or chunk-linking at index time.")


--- Fix: retrieve enough chunks (k=2) to cover both facts ---

Retrieved (k=2): ['split-fact-A', 'split-fact-B']
Answer: "Project Kestrel was founded in 2019 as an internal research initiative. Project Kestrel's founding team consisted of Priya Nair and Tom Alcock."

Raising k isn't a universal fix (it also raises noise and cost) -- the real lesson is
that naive top-k retrieval has no way to know an answer needs multiple chunks at all;
production systems handle this with larger k plus a reranker, or chunk-linking at index time.


### Break It 4: Near-duplicate confusable document wins

The confusable documents generated earlier are, by construction, nearly identical to a real
passage except for one swapped fact. A retriever can rank the wrong one first because the
similarity score is almost the same.

In [ ]:
print("--- Bug: a query about the original fact retrieves the confusable instead ---\n")
confusable_hits = 0
for c in confusables:
    original = next(d for d in docs if d["doc_id"] == c["confusable_of"])
    matching_qas = [qa for qa in qa_pairs if qa["gold_doc_id"] == c["confusable_of"]]
    if not matching_qas:
        continue
    qa = matching_qas[0]
    top_1 = tfidf_retriever.retrieve(qa["question"], k=1)
    if top_1 == [c["doc_id"]]:
        confusable_hits += 1
        print(f"Question: {qa['question']}")
        print(f"Retrieved: {top_1} (the CONFUSABLE, not {c['confusable_of']!r})")
        print(f"Synthetic change in play: {c['synthetic_change']}")
        wrong_answer = generate(qa["question"], [doc_lookup[c["doc_id"]]])
        print(f"Answer (confidently uses the swapped fact): {wrong_answer!r}\n")

print(f"{confusable_hits} of {len(confusables)} confusable documents won top-1 over their original at least once.")


--- Bug: a query about the original fact retrieves the confusable instead ---

Question: Theories on imperialism use which country as a model?
Retrieved: ['squad-027-confusable'] (the CONFUSABLE, not 'squad-027')
Synthetic change in play: name 'Minister Benjamin' -> 'Sarah Thompson'
Answer (confidently uses the swapped fact): 'In anglophone academic works, theories regarding imperialism are often based on the British experience.'

Question: What is the Mongolian name of the first Mongolian laws codified in writing?
Retrieved: ['squad-003-confusable'] (the CONFUSABLE, not 'squad-003')
Synthetic change in play: name 'Genghis Khan' -> 'Sarah Thompson'
Answer (confidently uses the swapped fact): 'He is also given credit for the introduction of the traditional Mongolian script and the creation of the Ikh Zasag (Great Administration), the first written Mongolian law.'

Question: How common was the form of corporal punishment in the past?
Retrieved: ['squad-011-confusable'] (the CONFUSABLE, n

In [ ]:
print("--- Fix: catch near-duplicates at index time instead of relying on retrieval to sort it out ---\n")
clean_docs = [d for d in all_docs if "confusable_of" not in d]
clean_retriever = TfidfRetriever(clean_docs)

for c in confusables[:1]:
    matching_qas = [qa for qa in qa_pairs if qa["gold_doc_id"] == c["confusable_of"]]
    if matching_qas:
        qa = matching_qas[0]
        top_1 = clean_retriever.retrieve(qa["question"], k=1)
        print(f"Question: {qa['question']}")
        print(f"Retrieved: {top_1} -- correct: {top_1 == [c['confusable_of']]}")

print("\nThe real fix for this class of bug is upstream of retrieval entirely: catch and merge")
print("or flag near-duplicates during ingestion (the deduplication step built earlier in this")
print("chapter), rather than hoping the retriever always breaks the tie correctly at query time.")


--- Fix: catch near-duplicates at index time instead of relying on retrieval to sort it out ---

Question: Theories on imperialism use which country as a model?
Retrieved: ['squad-027'] -- correct: True

The real fix for this class of bug is upstream of retrieval entirely: catch and merge
or flag near-duplicates during ingestion (the deduplication step built earlier in this
chapter), rather than hoping the retriever always breaks the tie correctly at query time.


### Break It 5: Offline metrics improve, online satisfaction drops

A simulated case: a retriever change that objectively improves the offline eval set (higher
recall@k on precisely-phrased SQuAD questions), evaluated against a separate sample of
conversational, imprecise user queries. This is the offline-vs-online divergence every
production ML/RAG team eventually hits.

**Hint:** The v2 retriever adds bigrams (2-word phrases), which rewards exact phrase matches
more heavily. This helps on benchmark questions (which reuse the passage's own words) and
hurts on real user queries (which rarely contain exact 2-word phrases from the source).

**Production impact:** shipping on the strength of the offline number alone, without checking
a held-out online signal, is the actual mistake. An improved benchmark score is not proof a
change should ship.

In [ ]:
class BigramTfidfRetriever:
    '''v2: adds bigrams (2-word phrases) to the vectorizer on top of v1's unigrams-only
    TF-IDF. This rewards exact phrase matches more heavily -- a real, mechanistic change,
    not just a relabeling -- which tends to help on precisely-phrased benchmark questions
    and tends to hurt on loosely-phrased, conversational real queries where exact 2-word
    phrase overlap with the source passage is much rarer.'''
    def __init__(self, docs: list):
        self.doc_ids = [d["doc_id"] for d in docs]
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.doc_vectors = self.vectorizer.fit_transform([d["text"] for d in docs])

    def retrieve(self, query: str, k: int = 3) -> list:
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.doc_vectors)[0]
        ranked = sims.argsort()[::-1][:k]
        return [self.doc_ids[i] for i in ranked]


v1_retriever = tfidf_retriever  # unigrams, already built above
v2_retriever = BigramTfidfRetriever(all_docs)

v1_offline = eval_metrics.evaluate_retrieval(eval_queries, v1_retriever.retrieve, k=3, **metric_fns)
v2_offline = eval_metrics.evaluate_retrieval(eval_queries, v2_retriever.retrieve, k=3, **metric_fns)

print("Offline eval (real, precisely-phrased SQuAD questions):")
print(f"  v1 (unigrams) recall@k: {v1_offline['recall@k']:.3f}")
print(f"  v2 (+ bigrams) recall@k: {v2_offline['recall@k']:.3f}")


Offline eval (real, precisely-phrased SQuAD questions):
  v1 (unigrams) recall@k: 0.945
  v2 (+ bigrams) recall@k: 0.964


In [ ]:
REALISTIC_QUERIES = [
    "hey quick question, who started anthropic again?",
    "can you remind me what react stands for in the agent context",
    "what happened with genghis khan and the shah's army",
    "kublai khan's advisors -- who were they",
    "corporal punishment in schools -- how common was it historically",
]


def simulated_satisfaction(retriever_name: str, query: str) -> float:
    '''Deliberately hand-constructed, deterministic stand-in for a real user-satisfaction
    signal (e.g. thumbs-up rate on live traffic) -- this is NOT derived from the retrievers'
    actual behavior above, and it is not meant to be: real online/offline divergence isn't
    always mechanistically traceable to one code change the way this notebook's other
    break-it bugs are, which is exactly what makes it dangerous in practice -- a team ships
    v2 because recall@k went up, and the online signal (which most teams check far less
    often than an offline eval suite) quietly moves the other way for unrelated or
    hard-to-pin-down reasons. Modeled here as v1 scoring higher than v2 on average, fixed
    seed per query for reproducibility.'''
    rng = random.Random(hash((retriever_name, query)) % (2**32))
    base = 0.78 if retriever_name == "v1" else 0.58
    return max(0.0, min(1.0, base + rng.uniform(-0.07, 0.07)))


v1_satisfaction = [simulated_satisfaction("v1", q) for q in REALISTIC_QUERIES]
v2_satisfaction = [simulated_satisfaction("v2", q) for q in REALISTIC_QUERIES]

print("Simulated online satisfaction (held-out, realistic/conversational queries):")
print(f"  v1 (unigrams) avg satisfaction:  {sum(v1_satisfaction) / len(v1_satisfaction):.3f}")
print(f"  v2 (+ bigrams) avg satisfaction: {sum(v2_satisfaction) / len(v2_satisfaction):.3f}")
print()
print("v2 won the offline benchmark above (higher recall@k) and loses here. The offline eval")
print("set's precisely-phrased benchmark questions and real users' loose, conversational")
print("phrasing are not the same distribution -- a change that helps on one can hurt on the")
print("other, and it doesn't always show up as a clean mechanistic story you can point to.")
print("Shipping on the strength of the offline number alone, without ever checking a held-out")
print("online signal, is the actual mistake this scenario is about.")


Simulated online satisfaction (held-out, realistic/conversational queries):
  v1 (unigrams) avg satisfaction:  0.765
  v2 (+ bigrams) avg satisfaction: 0.543

v2 won the offline benchmark above (higher recall@k) and loses here. The offline eval
set's precisely-phrased benchmark questions and real users' loose, conversational
phrasing are not the same distribution -- a change that helps on one can hurt on the
other, and it doesn't always show up as a clean mechanistic story you can point to.
Shipping on the strength of the offline number alone, without ever checking a held-out
online signal, is the actual mistake this scenario is about.


## Section 7: Interview Q&A

### Question 1: "My RAG system returns wrong answers. How do you diagnose which stage failed?"

Start with retrieval metrics. Run precision@k and recall@k on the failing queries. If both
are near zero, the retriever never found the right document -- fix indexing, query expansion,
or the embedding model. If recall@k is decent but MRR is low, the right document was
retrieved but buried -- add a reranker or adjust scoring weights. If all retrieval metrics
are fine, the problem is generation: the model saw the right context and ignored it. Check
faithfulness scores and adjust the generation prompt or add a faithfulness filter.

The critical insight is that fixing the wrong stage wastes effort. Re-tuning the generator
when retrieval is broken changes nothing because the model never saw the right context.

### Question 2: "Why would you use BM25 over embedding search, or vice versa?"

BM25 handles exact-match queries that embeddings structurally cannot. An arbitrary identifier
(SKU-1234, a part number, an internal code) was minted by a database and appeared in no
training corpus, so the embedding model has no learned representation for it. BM25 matches it
literally and ranks the right document first.

Embeddings handle synonyms and paraphrases that BM25 structurally cannot. "Car" and
"automobile" share no characters, so BM25 gives them zero score. An embedding model maps them
to nearby vectors because they appear in similar training contexts.

The standard production answer is both: BM25 for exact-match, embeddings for semantic, with
reciprocal rank fusion to merge the ranked lists. The precondition is that both retrievers
are individually strong. Fusing a strong retriever with a weak one moves results toward the
weak one.

### Question 3: "Chunk overlap costs index space. How much, and is it worth it?"

Halving the step roughly doubles the chunk count. The recovery rate experiment in this
chapter measured it: going from 0 overlap to 160 (out of 200 chunk size) raises recovery
of boundary-spanning facts from ~50% to ~95%, and the index grows about 3.5x.

Whether that trade is worth it depends on two things: how expensive your vectors are to
store and query, and how badly a split fact hurts your downstream task. A customer-facing
search where a split fact means a wrong answer has a different tolerance than an internal
analytics pipeline where approximate answers are acceptable.

### Question 4: "Your offline metrics look great but production quality is poor. What happened?"

The offline eval set and real user queries come from different distributions. Benchmark
questions are typically well-formed and precise (they often reuse words from the source
passage). Real user queries are conversational, imprecise, and use different vocabulary.

A change that helps on precisely-phrased benchmark questions (like adding bigram matching)
can hurt on loosely-phrased real queries. The fix is to maintain a held-out evaluation set
that mirrors real traffic patterns, not just benchmark distributions, and to check it before
shipping.

### Question 5: "How do you handle near-duplicate documents in your retrieval corpus?"

Catch them at ingest time, not at query time. Compute pairwise TF-IDF cosine similarity
between chunks and drop any chunk above a similarity threshold (0.85 works in practice) to
an already-kept chunk. This prevents confusable documents from crowding out the correct one
in retrieval results.

The alternative -- hoping the retriever always breaks the tie correctly at query time -- fails
because near-duplicates have almost identical similarity scores to the query, so which one
ranks first depends on implementation details (tie-breaking order, floating-point rounding)
rather than correctness.

### Cold-diagnosis exercise

For each scenario below, decide: is this a retrieval problem or a generation problem, and
which metric proves it? Attempt from memory, then check
`solutions/ch03_rag_evaluation_answers.md`.

Use `drill.check(n)` after writing your answer to see it alongside the model answer.
`drill.reveal(n)` shows the model answer without requiring your own attempt first.

In [ ]:
from agentlib.self_check import drill as open_drill

drill = open_drill(3)
drill.questions()

#### Answering these

Write your answer into the slot for each question, run the cell, then use `drill.check(n)`
to compare against the model answer. `check(n)` will not show you the model answer until you
have written one of your own.

In [ ]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

# Question 5
drill.attempt(5, '''
(Your answer here.)
''')

# Question 6
drill.attempt(6, '''
(Your answer here.)
''')

# Question 7
drill.attempt(7, '''
(Your answer here.)
''')

# Question 8
drill.attempt(8, '''
(Your answer here.)
''')

# Question 9
drill.attempt(9, '''
(Your answer here.)
''')

print()
drill.status()

In [ ]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

#### CEO explanation exercise

Write, in plain language suitable for a non-technical CEO, why an AI system that uses RAG
can still hallucinate. No jargon (no "retrieval," "embeddings," "faithfulness").

In [ ]:
my_ceo_explanation = '''
(Write your answer to question 6 here.)
'''

print(my_ceo_explanation)


## Section 8: References

1. Rajpurkar, P., Zhang, J., Lopyrev, K., & Liang, P. (2016). "SQuAD: 100,000+ Questions
   for Machine Comprehension of Text." arXiv:1606.05250.
2. Robertson, S. E., & Zaragoza, H. (2009). "The Probabilistic Relevance Framework: BM25
   and Beyond." Foundations and Trends in Information Retrieval 3(4).
3. Johnson, J., Douze, M., & Jegou, H. (2017). "Billion-scale similarity search with GPUs."
   arXiv:1702.08734 (FAISS).
4. Es, S., et al. (2024). "RAGAS: Automated Evaluation of Retrieval Augmented Generation."
   arXiv:2309.15217.
5. scikit-learn TF-IDF documentation. https://scikit-learn.org/stable/modules/feature_extraction.html
6. FAISS documentation. https://github.com/facebookresearch/faiss
7. spaCy en_core_web_md model. https://spacy.io/models/en

### Related chapters

- **Chapter 1** (Fundamentals): grounding concept, why models hallucinate
- **Chapter 5** (Cost, Performance): measuring the actual cost of retrieval calls,
  context window usage from raising k
- **Chapter 7** (Tool Integration): retrieval as a tool, schema validation of results

## Next: Chapter 4: Production Reliability

This chapter treated retrieval and generation as if they always run correctly when called.
Chapter 4 is about what happens when they don't: stale caches, flaky tools, and cascading
failures -- the gap between a demo and something that survives production traffic.